### Python to C++ Code Converter

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

import subprocess
# This module allows Python code to execute operating system commands, 
# run external applications, and spawn new system processes directly from within your notebook cells
# Common uses include:
# Running terminal/shell commands from Python variables.
# Executing other scripts or tools (e.g., calling git, ffmpeg, curl, or running a C++/Rust executable).
# Capturing standard output (stdout) and errors (stderr) from system commands directly into Python strings for further processing


In [ ]:
# base URLs and API keys

load_dotenv(override=True)

groq_base_url = os.getenv('GROQ_BASE_URL')
groq_api_key = os.getenv('GROQ_API_KEY')

gemini_base_url = os.getenv('GEMINI_BASE_URL')
gemini_api_key = os.getenv('GEMINI_API_KEY')

if groq_api_key:
    print(f'Groq API Key found and starts with {groq_api_key[0:3]}')
else:
    print('Groq API Key not found')

if gemini_api_key:
    print(f'Gemini API Key found and starts with {gemini_api_key[0:3]}')
else:
    print('Gemini API Key not found')

Groq API Key found and starts with gsk
Gemini API Key found and starts with AQ.


In [ ]:
# clients

groq = OpenAI(base_url = groq_base_url, api_key = groq_api_key)
gemini = OpenAI(base_url = gemini_base_url, api_key = gemini_api_key)

In [23]:
# models

gpt_model = 'openai/gpt-oss-20b'
qwen_model = 'qwen/qwen3.6-27b'
gemini_model = 'gemini-3.6-flash'

#### Task

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

In [ ]:
# for system information

from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '25.5.0',
  'version': 'Darwin Kernel Version 25.5.0: Tue Jun  9 22:26:46 PDT 2026; root:xnu-12377.121.10~1/RELEASE_ARM64_T8103',
  'kernel': '25.5.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin25.5.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M1',
  'cores_logical': 8,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [ ]:
# to generate compile command and run command from the system info

message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = gemini.chat.completions.create(model=gemini_model, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

**No, you do not need to install any C++ compiler.** 

Your system already has Apple Clang (Xcode Command Line Tools) installed and ready to go (`clang++` / `g++`).

---

### Python Code Setup

To achieve the **fastest possible runtime performance** on your Apple M1 CPU, use `-O3` (maximum compiler optimization) along with `-mcpu=native` (targets the specific architecture and instruction set of your M1 chip).

Here is exactly what you should use for `compile_command` and `run_command`:

```python
import subprocess

# Compile command: Apple Clang with aggressive speed optimizations targeting Apple M1
compile_command = ["clang++", "-O3", "-mcpu=native", "main.cpp", "-o", "main"]

compile_result = subprocess.run(
    compile_command, check=True, text=True, capture_output=True
)

# Run command: Executable produced in the current working directory
run_command = ["./main"]

run_result = subprocess.run(
    run_command, check=True, text=True, capture_output=True
)

print(run_result.stdout)
```

### Flag Breakdown:
* `clang++`: The C++ compiler driver for Apple Clang installed on your system.
* `-O3`: Enables high-level loop, vectorization, and speed optimizations.
* `-mcpu=native`: Instructs Clang to generate code tailored specifically to your Apple M1 CPU core features.
* `-o main`: Names the generated executable file `main`.

In [ ]:
# Set-up to execute Cpp code from the python environment itself

# compile command and run command

compile_command = ["clang++", "-O3", "-mcpu=native", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [28]:
# system prompt

system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

In [ ]:
# user prompt

def user_prompt(python_code):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python_code}
```
"""

In [39]:
# packaging the complete message

def messages(python_code):
    return [
        {'role':'system', 'content':system_prompt},
        {'role':'user', 'content':user_prompt(python_code)}
    ]

In [33]:
# write to main.cpp

def write_cpp(cpp):
    with open("main.cpp", 'w', encoding = 'utf-8') as f:
        f.write(cpp)

In [34]:
# to port the code from Python to C++
 
def port(client, model, python_code):
    response = client.chat.completions.create(
        model = model,
        messages = messages(python_code)
    )

    reply = response.choices[0].message.content
    reply = reply.replace('```cpp', '').replace('```', '')
    write_cpp(reply)

In [35]:
# Sample python code to calulate pi value

pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [36]:
# setting the environment to run the python code inside the notebook natively

def run_python_code(code):
    globals = {"__builtins__":__builtins__}
    exec(code, globals)

In [ ]:
# to run the python code to calculate pi-value along with time taken to compute using Python

run_python_code(pi)

Result: 3.141592656089
Execution Time: 21.503796 seconds


In [40]:
# porting the python code to Cpp using Gemini 3.6 flash

port(gemini, gemini_model, pi)

In [41]:
def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)


In [42]:
compile_and_run()

Result: 3.141592656090
Execution Time: 0.021660 seconds

Result: 3.141592656090
Execution Time: 0.021282 seconds

Result: 3.141592656090
Execution Time: 0.018400 seconds

